# Week 2 lecture walkthrough: simplices, boundary maps and homology over $\mathbb F_2$

This is the live worked example, built around the outline and filled triangle, square loop and hollow tetrahedron. It follows the conceptual argument of the slides through prediction, reveal and interpretation. It is not intended as a line-by-line answer key to the participant practical.

**Resource boundary.** The [reference notes](index.qmd) define the objects and explain the algebra. The [slides](slides.qmd) organise the live argument. This notebook works every central example to completion. The [participant practical](lab.ipynb) retains the same small complexes but leaves matrix reading, checks and interpretation to students.

**Lecture map.** Begin with the same 1-skeleton drawn as an outline and filled triangle. Reveal the chain spaces and boundary matrices only after predictions. Compare the square loop, then lift the argument by one dimension with the hollow and solid tetrahedron.

This practical makes the algebra visible before any TDA library is used. We will move through

$$K \longrightarrow C_p(K;\mathbb F_2) \xrightarrow{\partial_p} C_{p-1}(K;\mathbb F_2)
\longrightarrow Z_p/B_p = H_p(K;\mathbb F_2).$$

The examples are deliberately small: an outline triangle, a filled triangle, a square loop and a hollow tetrahedron.

**◇ Object check.** The simplicial complex $K$ is a collection of simplices. The chain space $C_p$ is a vector space whose chosen basis is the list of $p$-simplices. A matrix represents a boundary map only after those bases and their orders have been fixed.

**Presenter route.** Use the outline-to-filled triangle as the central reveal: the 1-skeleton stays fixed while $C_2$, $\partial_2$ and $H_1$ change.


**Prerequisite bridge.** Use the [algebra survival guide](algebra-survival-guide.qmd) if basis, kernel, image, rank or quotient are unfamiliar.


## Conventions and reading connection

The notebooks use **abstract simplicial complexes**: a simplex is represented by its set of vertices. A geometric realisation can place those simplices in Euclidean space, but the chain calculation uses only which faces belong to which simplices.

Every simplex here is non-empty, and all homology calculations use coefficients in $\mathbb F_2$. This matches the modulo-2 computational development in Edelsbrunner and Harer, Chapter IV, and complements Postol, Section 4.1. Other coefficient systems require orientation signs and may reveal torsion.

In [ ]:
from itertools import combinations
import numpy as np
import matplotlib.pyplot as plt

def simplex_key(simplex):
    return (len(simplex), tuple(simplex))

def close_under_faces(maximal_simplices):
    """Return all non-empty faces, grouped by dimension."""
    simplices = set()
    for maximal in maximal_simplices:
        maximal = tuple(sorted(maximal))
        for size in range(1, len(maximal) + 1):
            simplices.update(combinations(maximal, size))
    max_dim = max(len(s) - 1 for s in simplices)
    return {p: sorted((s for s in simplices if len(s) == p + 1))
            for p in range(max_dim + 1)}

def boundary_matrix(K, p):
    """Matrix of ∂_p over F_2; columns and rows use the displayed bases."""
    columns = K.get(p, [])
    rows = K.get(p - 1, []) if p > 0 else []
    matrix = np.zeros((len(rows), len(columns)), dtype=np.uint8)
    row_index = {simplex: i for i, simplex in enumerate(rows)}
    if p > 0:
        for j, simplex in enumerate(columns):
            for face in combinations(simplex, p):
                matrix[row_index[face], j] = 1
    return matrix

def rank_mod2(matrix):
    """Gaussian elimination over F_2."""
    A = np.array(matrix, dtype=np.uint8, copy=True) % 2
    rank = row = 0
    for col in range(A.shape[1]):
        candidates = np.flatnonzero(A[row:, col])
        if len(candidates) == 0:
            continue
        pivot = row + candidates[0]
        A[[row, pivot]] = A[[pivot, row]]
        for other in range(A.shape[0]):
            if other != row and A[other, col]:
                A[other] ^= A[row]
        rank += 1
        row += 1
        if row == A.shape[0]:
            break
    return rank

def homology_table(K):
    """Dimensions of C_p, Z_p, B_p and H_p over F_2."""
    result = []
    max_dim = max(K)
    for p in range(max_dim + 1):
        dim_C = len(K.get(p, []))
        rank_dp = rank_mod2(boundary_matrix(K, p)) if p > 0 else 0
        rank_next = rank_mod2(boundary_matrix(K, p + 1)) if p < max_dim else 0
        dim_Z = dim_C - rank_dp
        dim_B = rank_next
        result.append((p, dim_C, dim_Z, dim_B, dim_Z - dim_B))
    return result

def report(name, K):
    print(name)
    for p in sorted(K):
        print(f"  basis C_{p}: {K[p]}")
    print("  p | dim C_p | dim Z_p | dim B_p | beta_p")
    for row in homology_table(K):
        print(" ", " | ".join(map(str, row)))

def check_boundary_squared(K):
    for p in range(2, max(K) + 1):
        product = (boundary_matrix(K, p - 1) @ boundary_matrix(K, p)) % 2
        assert not product.any(), f"Boundary squared failed in dimension {p}"
    return True

outline_triangle = close_under_faces([(0, 1), (1, 2), (0, 2)])
filled_triangle = close_under_faces([(0, 1, 2)])
square_loop = close_under_faces([(0, 1), (1, 2), (2, 3), (0, 3)])
hollow_tetrahedron = close_under_faces([(0, 1, 2), (0, 1, 3),
                                        (0, 2, 3), (1, 2, 3)])

complexes = {
    "outline triangle": outline_triangle,
    "filled triangle": filled_triangle,
    "square loop": square_loop,
    "hollow tetrahedron": hollow_tetrahedron,
}
print("Defined four finite simplicial complexes over F_2.")

## 1. Observe: lecture demonstration

**Presenter cue.** Show the object before the calculation. Ask the room to separate what is given from what will be constructed.

The first three complexes are drawn below. The hollow tetrahedron is the union of its four triangular faces. It does **not** contain the solid tetrahedral 3-simplex.

Before computing, distinguish the visible drawing from the mathematical object. In particular, the outline and filled triangles have the same vertices and edges, hence the same 1-skeleton, but they are different simplicial complexes.

In [ ]:
positions_2d = {
    "outline triangle": {0:(0,0), 1:(1,0), 2:(0.5,0.87)},
    "filled triangle": {0:(0,0), 1:(1,0), 2:(0.5,0.87)},
    "square loop": {0:(0,0), 1:(1,0), 2:(1,1), 3:(0,1)},
}
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
for ax, (name, K) in zip(axes, list(complexes.items())[:3]):
    pos = positions_2d[name]
    if K.get(2):
        for face in K[2]:
            ax.fill(*zip(*(pos[v] for v in face)), alpha=0.25, color='tab:blue')
    for edge in K.get(1, []):
        ax.plot(*zip(*(pos[v] for v in edge)), color='black')
    for v, xy in pos.items():
        ax.scatter(*xy, s=80, zorder=3)
        ax.text(xy[0], xy[1] + 0.08, str(v), ha='center')
    ax.set_title(name)
    ax.set_aspect('equal'); ax.axis('off')
plt.show()
print("The hollow tetrahedron contains four triangular faces but no 3-simplex.")

## 2. Predict: lecture demonstration

**Presenter cue.** Pause here and collect at least two predictions before revealing any output.

Without running the homology calculations, complete this table. Count a connected component as an $H_0$ class.

| Complex | Expected $\beta_0$ | Expected $\beta_1$ | Expected $\beta_2$ | Which simplex could fill the visible cycle? |
|---|---:|---:|---:|---|
| Outline triangle |  |  |  |  |
| Filled triangle |  |  |  |  |
| Square loop |  |  |  |  |
| Hollow tetrahedron |  |  |  |  |

Also predict whether adding the triangular 2-simplex to the outline triangle changes $C_0$, $C_1$, $C_2$, or more than one of them.

**Worked prediction.** All four complexes are connected, so $\beta_0=1$. The outline triangle and square loop each have one unfilled 1-cycle. The filled triangle has no nonzero higher-dimensional homology. The hollow tetrahedron has one 2-dimensional surface cycle. Thus the expected Betti vectors are $(1,1)$, $(1,0,0)$, $(1,1)$ and $(1,0,1)$. A 2-simplex fills each planar loop; a 3-simplex fills the tetrahedral surface.

## 3. Implement: lecture demonstration

**Reveal.** Run one cell at a time. Name the domain, codomain, complex, module or summary before interpreting its values.

Run the setup cell. `close_under_faces` enforces downward closure. `boundary_matrix` creates the matrix of $\partial_p$ over $\mathbb F_2$: an entry is 1 when the row simplex is a codimension-one face of the column simplex.

Because $-1=1$ in $\mathbb F_2$, orientations and signs disappear. This is convenient, but it is a coefficient choice, not a property of homology in general.

### A. Read a boundary matrix

In [ ]:
d1_outline = boundary_matrix(outline_triangle, 1)
print('rows, basis C_0:', outline_triangle[0])
print('columns, basis C_1:', outline_triangle[1])
print(d1_outline)
triangle_edge_cycle = np.ones(3, dtype=np.uint8)
print('boundary of all three edges:', (d1_outline @ triangle_edge_cycle) % 2)

The map is $\partial_1:C_1\to C_0$. Each column contains the two endpoints of its edge. Adding all columns gives zero because every vertex occurs twice. The edge sum is therefore in $Z_1$.

### B. Check that boundaries are cycles

In [ ]:
d1_filled = boundary_matrix(filled_triangle, 1)
d2_filled = boundary_matrix(filled_triangle, 2)
print('(∂_1 ∂_2) =')
print((d1_filled @ d2_filled) % 2)
for name, K in complexes.items():
    print(name, check_boundary_squared(K))

The zero product states that the boundary of the triangular face has no boundary. More generally, $\partial_p\partial_{p+1}=0$, so every vector in $\operatorname{im}\partial_{p+1}=B_p$ lies in $\ker\partial_p=Z_p$.

### C. Compute homology dimensions

In [ ]:
report('outline triangle', outline_triangle)
report('square loop', square_loop)

Both are connected and have one independent edge cycle with no 2-chain available to bound it. Their visible geometry differs, but their Betti numbers over $\mathbb F_2$ agree.

## 4. Compare: lecture demonstration

In [ ]:
report('outline triangle', outline_triangle)
report('filled triangle', filled_triangle)

Adding the face leaves $C_0$, $C_1$ and $\partial_1$ unchanged. It creates a one-dimensional $C_2$ and a nonzero $\partial_2$. The old edge cycle remains in $Z_1$, but it is now also in $B_1$, so its class is zero in $H_1=Z_1/B_1$. This is why the complex must not be confused with its 1-skeleton.

### The hollow tetrahedron

In [ ]:
faces = hollow_tetrahedron[2]
hollow_surface = np.ones(len(faces), dtype=np.uint8)
d2_hollow = boundary_matrix(hollow_tetrahedron, 2)
print('boundary of the four-face surface:', (d2_hollow @ hollow_surface) % 2)
report('hollow tetrahedron', hollow_tetrahedron)
solid_tetrahedron = close_under_faces([(0, 1, 2, 3)])
report('solid tetrahedron', solid_tetrahedron)
print('boundary of its 3-simplex:', boundary_matrix(solid_tetrahedron, 3).ravel())

Every edge of the hollow surface occurs in two faces, so the face sum is a 2-cycle. With no 3-simplex, $B_2=0$ and $\beta_2=1$. Adding the solid tetrahedron creates $C_3\cong\mathbb F_2$; its boundary is exactly the four-face cycle, which becomes zero in $H_2$.

### D. Use Euler characteristic as a cross-check

For a finite complex,

$$\chi(K)=\sum_p(-1)^p\dim C_p=\sum_p(-1)^p\beta_p.$$

Check both sides for all four examples. Explain why agreement is useful but does not prove that every individual Betti number is correct.

In [ ]:
def euler_from_simplices(K):
    return sum((-1)**p * len(K.get(p, [])) for p in K)

def euler_from_betti(K):
    return sum((-1)**p * row[-1] for p, row in enumerate(homology_table(K)))

for name, K in complexes.items():
    left = euler_from_simplices(K)
    right = euler_from_betti(K)
    print(f"{name}: simplex count gives {left}; Betti numbers give {right}")
    assert left == right

Agreement checks the alternating total of the computed dimensions. It cannot locate an error by degree: simultaneous errors in adjacent Betti numbers can cancel in the alternating sum. Boundary ranks and $\partial^2=0$ must still be checked directly.

### Quotient checkpoint: worked

In the filled triangle, $z-0=z=\partial_2(012)\in B_1$, so $z\sim0$ and $[z]=[0]$. More generally, if $z'-z$ is the boundary of an available two-chain, then $z'\sim z$ and both cycles represent the same homology class.

A homology class is one **equivalence** class containing many cycle **representatives**. The quotient forgets changes produced by adding boundaries.

## 5. Interpret: lecture demonstration

**Presenter close.** Ask what the example supports, what information was discarded and which stronger claim would be unjustified.

1. The outline cycle is not a boundary because $C_2=0$. In the filled triangle the same chain equals the boundary of the 2-simplex.
2. A homology class is an equivalence class of cycles modulo boundaries. A drawn loop is only one possible representative.
3. The tetrahedral 3-simplex fills the hollow tetrahedron's $H_2$ class.
4. The matrix arithmetic, cancellation of duplicate faces, and omission of orientation signs here use $\mathbb F_2$. The definitions of cycles, boundaries and quotient homology persist with other coefficients, but the algebra can change.
5. Filling a 3-clique asserts that three pairwise relations constitute a higher-order unit. That is a modelling decision, not information contained in the graph alone.

**† Qualification.** These calculations describe the topology of the constructed simplicial complex. Whether that complex faithfully represents a dynamical system is a separate scientific and modelling question.

## Lecture close

Return to the final slide questions.

1. Name the observed or starting object.
2. Name every constructed object used in this walkthrough.
3. Identify the single modelling decision that drove the central comparison.
4. State one conclusion supported by the calculation and one conclusion it cannot establish.

**Take-forward example.** The purpose of the outline and filled triangle, square loop and hollow tetrahedron is to make simplices, boundary maps and homology over $\mathbb F_2$ concrete. The example is deliberately small or synthetic so that the construction remains inspectable.